In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
import kagglehub


%matplotlib inline

In [ ]:

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Read the dataset Q1_data.csv using read_csv()
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Inspect the first few rows using head()
df.head()

In [ ]:
# Task 3: Display dataset information using info()
df.info()

In [ ]:
# Task 4: Show statistical description using describe()
df.describe()

In [ ]:
# Task 5: Plot the target distribution (delivery_time)
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Drop the 'Order_ID' column from the data
df.drop('Order_ID', axis=1, inplace= True)

In [ ]:
float(df['Courier_Experience_yrs'].mean())

In [ ]:
# Task 2: Handle missing values appropriately
# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

print('data befor clean: ')
check_missing_values(df)

df['Courier_Experience_yrs'].fillna(float(df['Courier_Experience_yrs'].mean()), inplace=True)
df.dropna(subset=['Delivery_Time'], inplace=True)
df.fillna('UnKnown',inplace=True)
print()
print('data after clean: ')
check_missing_values(df)

In [ ]:
# Task 3: Check and remove duplicates if any exist
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=["object"]).columns


for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))


In [ ]:
# Task 5: Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler

features = df.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()


In [ ]:
# Task 6: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
# our task is reg not classificaion

In [ ]:
# Task 1: Split the dataset into features (X) and target (y)
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)



models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
}

scores = 0
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    print(f'mae = {mae}')
    scores += mae

print(f'Avg of all scores = {scores/n_splits}')





In [ ]:
# Task 1: Plot feature importance from your trained model
from sklearn.linear_model import Ridge, Lasso
coeffs = {}

coeffs['Lasso'] = models['Random Forest Regressor'].coef_
coeffs['Ridge'] = models['Random Forest Regressor'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Plot predicted delivery time histogram
y_pred = pd.DataFrame(y_pred)
y_pred.hist()

In [ ]:
# Task Bonus: Write your code here: